Efficiency Trade-offs Analysis

Research Question: Which quantization offers the best trade-off between
quality and efficiency for practical RAG deployment?

Analysis Dimensions:
- Latency: Generation speed and time-to-first-token
- Memory: Allocated, reserved, peak usage
- Model Size: Disk footprint and compression ratios
- Quality-Efficiency Pareto: F1 vs speed, F1 vs memory

Setup and Imports

In [73]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

Configuration

In [74]:
# Path to your generated results
RESULTS_DIR = Path('/kaggle/input/generation-sets')  # Update this path
OUTPUT_DIR = Path('/kaggle/working/efficiency_analysis')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model configurations to analyze
CONFIGS = [
    'fp16_base', 'fp16_instruct',
    'awq_base', 'awq_instruct',
    'nf4_base', 'nf4_instruct',
    'gptq_base', 'gptq_instruct'
]

# Quantization methods
QUANT_METHODS = ['fp16', 'awq', 'nf4', 'gptq']

logger.info(f"Results directory: {RESULTS_DIR}")
logger.info(f"Output directory: {OUTPUT_DIR}")

2025-12-28 23:24:27,255 - INFO - Results directory: /kaggle/input/generation-sets
2025-12-28 23:24:27,256 - INFO - Output directory: /kaggle/working/efficiency_analysis


Data Loading Functions

In [75]:
def load_generation_file(config_name: str, set_name: str) -> Dict:
    """Load a single generation results file."""
    filename = f"{config_name}_{set_name}_complete.json"
    filepath = RESULTS_DIR / filename
    
    if not filepath.exists():
        logger.warning(f"File not found: {filename}")
        return None
    
    try:
        with open(filepath, 'r') as f:
            data = json.load(f)
        logger.info(f"Loaded: {filename}")
        return data
    except Exception as e:
        logger.error(f"Failed to load {filename}: {e}")
        return None

def load_all_generation_data() -> Dict[str, Dict]:
    """Load all generation data files."""
    all_data = {}
    
    for config in CONFIGS:
        config_data = {}
        
        for set_name in ['set_a', 'set_b', 'set_c', 'set_d']:
            data = load_generation_file(config, set_name)
            if data is not None:
                config_data[set_name] = data
        
        if config_data:
            all_data[config] = config_data
            logger.info(f"Loaded {len(config_data)} sets for {config}")
    
    logger.info(f"Total configurations loaded: {len(all_data)}")
    return all_data

Metric Extraction Functions

In [76]:
def extract_generation_times(data: Dict) -> List[float]:
    """Extract generation times from results data."""
    times = []
    
    if 'samples' not in data:
        return times
    
    for sample in data['samples']:
        if 'rag_generation_time_ms' in sample:
            times.append(sample['rag_generation_time_ms'])
        
        if 'no_rag_generation_time_ms' in sample:
            times.append(sample['no_rag_generation_time_ms'])
        
        if 'clean_generation_time_ms' in sample:
            times.append(sample['clean_generation_time_ms'])
        
        if 'distracted_generation_time_ms' in sample:
            times.append(sample['distracted_generation_time_ms'])
    
    return times

def compute_timing_statistics(times: List[float]) -> Dict:
    """Compute timing statistics from generation times."""
    if not times:
        return {'error': 'no timing data'}
    
    times_array = np.array(times)
    
    return {
        'mean_ms': float(np.mean(times_array)),
        'median_ms': float(np.median(times_array)),
        'std_ms': float(np.std(times_array)),
        'min_ms': float(np.min(times_array)),
        'max_ms': float(np.max(times_array)),
        'p95_ms': float(np.percentile(times_array, 95)),
        'p99_ms': float(np.percentile(times_array, 99)),
        'count': len(times)
    }

def estimate_tokens_per_second(mean_ms: float, avg_tokens: int = 64) -> float:
    """Estimate tokens per second from mean generation time."""
    if mean_ms <= 0:
        return 0.0
    return (avg_tokens / mean_ms) * 1000

def extract_context_lengths(data: Dict) -> List[int]:
    """Extract context lengths from samples."""
    lengths = []
    
    if 'samples' not in data:
        return lengths
    
    for sample in data['samples']:
        if 'context_length_tokens' in sample:
            lengths.append(sample['context_length_tokens'])
        elif 'actual_length_tokens' in sample:
            lengths.append(sample['actual_length_tokens'])
    
    return lengths

Quality Metric Extraction

In [77]:
def compute_f1_score(prediction: str, ground_truth: str) -> float:
    """Compute token-level F1 score."""
    pred_tokens = set(prediction.lower().split())
    gt_tokens = set(ground_truth.lower().split())
    
    if not pred_tokens and not gt_tokens:
        return 1.0
    if not pred_tokens or not gt_tokens:
        return 0.0
    
    common = len(pred_tokens & gt_tokens)
    if common == 0:
        return 0.0
    
    precision = common / len(pred_tokens)
    recall = common / len(gt_tokens)
    
    return 2 * precision * recall / (precision + recall)

def compute_exact_match(prediction: str, ground_truth: str) -> float:
    """Compute exact match score."""
    pred_clean = prediction.lower().strip()
    gt_clean = ground_truth.lower().strip()
    return 1.0 if pred_clean == gt_clean else 0.0

def extract_quality_metrics(data: Dict) -> Dict:
    """Extract quality metrics from generation data."""
    f1_scores = []
    em_scores = []
    
    if 'samples' not in data:
        return {'f1_mean': 0.0, 'em_mean': 0.0}
    
    for sample in data['samples']:
        gt = sample.get('ground_truth', '')
        
        if 'rag_prediction' in sample:
            pred = sample['rag_prediction']
            f1_scores.append(compute_f1_score(pred, gt))
            em_scores.append(compute_exact_match(pred, gt))
    
    return {
        'f1_mean': float(np.mean(f1_scores)) if f1_scores else 0.0,
        'f1_std': float(np.std(f1_scores)) if f1_scores else 0.0,
        'em_mean': float(np.mean(em_scores)) if em_scores else 0.0,
        'count': len(f1_scores)
    }

Model Size Estimation

In [78]:
def estimate_model_size(config_name: str) -> Dict:
    """Estimate model size based on configuration."""
    
    # Mistral 7B parameter count
    params = 7.24e9
    
    if 'fp16' in config_name:
        bytes_per_param = 2
        bits = 16
    else:  # AWQ, NF4, GPTQ all use 4-bit
        bytes_per_param = 0.5
        bits = 4
    
    size_bytes = params * bytes_per_param
    size_gb = size_bytes / (1024**3)
    
    return {
        'size_gb': float(size_gb),
        'params': int(params),
        'bits_per_param': float(bits)
    }

def estimate_memory_usage(config_name: str, context_length: int = 2048) -> Dict:
    """Estimate memory usage for generation."""
    
    # Mistral 7B specs
    num_layers = 32
    num_heads = 32
    head_dim = 128
    
    # Model weights
    size_info = estimate_model_size(config_name)
    model_memory_mb = size_info['size_gb'] * 1024
    
    # KV cache estimation
    bytes_per_element = 2  # FP16
    kv_cache_mb = (
        2 * num_layers * num_heads * context_length * head_dim * bytes_per_element
    ) / (1024**2)
    
    # Activation memory (rough estimate)
    activation_mb = model_memory_mb * 0.2
    
    total_mb = model_memory_mb + kv_cache_mb + activation_mb
    
    return {
        'model_mb': float(model_memory_mb),
        'kv_cache_mb': float(kv_cache_mb),
        'activation_mb': float(activation_mb),
        'total_mb': float(total_mb)
    }

Comprehensive Analysis

In [79]:
def analyze_configuration(config_name: str, config_data: Dict) -> Dict:
    """Perform comprehensive analysis for a single configuration."""
    logger.info(f"Analyzing: {config_name}")
    
    results = {
        'config_name': config_name,
        'quantization': config_name.split('_')[0],
        'variant': config_name.split('_')[1]
    }
    
    # Collect all generation times
    all_times = []
    for set_name, set_data in config_data.items():
        times = extract_generation_times(set_data)
        all_times.extend(times)
    
    # Timing statistics
    timing_stats = compute_timing_statistics(all_times)
    results['timing'] = timing_stats
    
    if 'error' not in timing_stats:
        # Compute tokens per second
        results['tokens_per_sec'] = estimate_tokens_per_second(
            timing_stats['mean_ms']
        )
    
    # Quality metrics from Set A
    if 'set_a' in config_data:
        quality = extract_quality_metrics(config_data['set_a'])
        results['quality'] = quality
    
    # Model size
    results['model_size'] = estimate_model_size(config_name)
    
    # Memory usage
    avg_context = 2048
    if 'set_a' in config_data:
        contexts = extract_context_lengths(config_data['set_a'])
        if contexts:
            avg_context = int(np.mean(contexts))
    
    results['memory'] = estimate_memory_usage(config_name, avg_context)
    
    return results

Load and Process All Data

In [80]:
logger.info("Loading generation data")
all_data = load_all_generation_data()

if not all_data:
    logger.error("No data loaded. Please check RESULTS_DIR path.")
else:
    logger.info(f"Loaded data for {len(all_data)} configurations")

2025-12-28 23:24:27,473 - INFO - Loading generation data
2025-12-28 23:24:27,484 - INFO - Loaded: fp16_base_set_a_complete.json
2025-12-28 23:24:27,545 - INFO - Loaded: fp16_base_set_b_complete.json
2025-12-28 23:24:27,556 - INFO - Loaded: fp16_base_set_c_complete.json
2025-12-28 23:24:27,565 - INFO - Loaded: fp16_base_set_d_complete.json
2025-12-28 23:24:27,565 - INFO - Loaded 4 sets for fp16_base
2025-12-28 23:24:27,574 - INFO - Loaded: fp16_instruct_set_a_complete.json
2025-12-28 23:24:27,608 - INFO - Loaded: fp16_instruct_set_b_complete.json
2025-12-28 23:24:27,617 - INFO - Loaded: fp16_instruct_set_c_complete.json
2025-12-28 23:24:27,626 - INFO - Loaded: fp16_instruct_set_d_complete.json
2025-12-28 23:24:27,627 - INFO - Loaded 4 sets for fp16_instruct
2025-12-28 23:24:27,635 - INFO - Loaded: awq_base_set_a_complete.json
2025-12-28 23:24:27,664 - INFO - Loaded: awq_base_set_b_complete.json
2025-12-28 23:24:27,673 - INFO - Loaded: awq_base_set_c_complete.json
2025-12-28 23:24:27,682

In [81]:
logger.info("Analyzing all configurations")
analysis_results = {}

for config_name, config_data in all_data.items():
    try:
        result = analyze_configuration(config_name, config_data)
        analysis_results[config_name] = result
    except Exception as e:
        logger.error(f"Failed to analyze {config_name}: {e}")

logger.info(f"Completed analysis for {len(analysis_results)} configurations")

2025-12-28 23:24:28,040 - INFO - Analyzing all configurations
2025-12-28 23:24:28,041 - INFO - Analyzing: fp16_base
2025-12-28 23:24:28,047 - INFO - Analyzing: fp16_instruct
2025-12-28 23:24:28,050 - INFO - Analyzing: awq_base
2025-12-28 23:24:28,052 - INFO - Analyzing: awq_instruct
2025-12-28 23:24:28,055 - INFO - Analyzing: nf4_base
2025-12-28 23:24:28,057 - INFO - Analyzing: nf4_instruct
2025-12-28 23:24:28,059 - INFO - Analyzing: gptq_base
2025-12-28 23:24:28,061 - INFO - Analyzing: gptq_instruct
2025-12-28 23:24:28,063 - INFO - Completed analysis for 8 configurations


Comparative Analysis

In [82]:
def create_comparison_table() -> pd.DataFrame:
    """Create comprehensive comparison table."""
    rows = []
    
    for config_name, result in analysis_results.items():
        row = {
            'config': config_name,
            'quantization': result['quantization'],
            'variant': result['variant']
        }
        
        # Timing
        if 'timing' in result and 'error' not in result['timing']:
            row['mean_ms'] = result['timing']['mean_ms']
            row['median_ms'] = result['timing']['median_ms']
            row['p95_ms'] = result['timing']['p95_ms']
        
        # Throughput
        if 'tokens_per_sec' in result:
            row['tokens_per_sec'] = result['tokens_per_sec']
        
        # Quality
        if 'quality' in result:
            row['f1_score'] = result['quality']['f1_mean']
            row['exact_match'] = result['quality']['em_mean']
        
        # Model size
        if 'model_size' in result:
            row['model_gb'] = result['model_size']['size_gb']
            row['bits_per_param'] = result['model_size']['bits_per_param']
        
        # Memory
        if 'memory' in result:
            row['total_memory_mb'] = result['memory']['total_mb']
            row['kv_cache_mb'] = result['memory']['kv_cache_mb']
        
        rows.append(row)
    
    df = pd.DataFrame(rows)
    
    if not df.empty and 'quantization' in df.columns:
        df = df.sort_values(['quantization', 'variant'])
    
    return df

comparison_df = create_comparison_table()
logger.info(f"Created comparison table with {len(comparison_df)} rows")

2025-12-28 23:24:28,085 - INFO - Created comparison table with 8 rows


In [83]:
print("\nCOMPREHENSIVE EFFICIENCY COMPARISON")
print("\nAll Metrics:")
print(comparison_df.to_string(index=False))


COMPREHENSIVE EFFICIENCY COMPARISON

All Metrics:
       config quantization  variant     mean_ms   median_ms       p95_ms  tokens_per_sec  f1_score  exact_match  model_gb  bits_per_param  total_memory_mb  kv_cache_mb
     awq_base          awq     base 6838.324790 5146.207516 14414.093766        9.359017  0.110974         0.05  3.371388             4.0      4245.261230        102.5
 awq_instruct          awq instruct 6489.486947 4879.961342 15673.978456        9.862105  0.109008         0.05  3.371388             4.0      4245.261230        102.5
    fp16_base         fp16     base 3841.159787 2738.639371  8235.077626       16.661634  0.094277         0.03 13.485551            16.0     16673.544922        102.5
fp16_instruct         fp16 instruct 3719.004849 2509.944030 10679.093142       17.208905  0.113013         0.04 13.485551            16.0     16673.544922        102.5
    gptq_base         gptq     base 2686.660095 2742.093071  5100.828715       23.821398  0.049319         0.

Quantization Method Aggregation

In [84]:
def aggregate_by_quantization() -> pd.DataFrame:
    """Aggregate metrics by quantization method."""
    quant_data = {method: [] for method in QUANT_METHODS}
    
    for config_name, result in analysis_results.items():
        quant = result['quantization']
        if quant in quant_data:
            quant_data[quant].append(result)
    
    rows = []
    for method, results in quant_data.items():
        if not results:
            continue
        
        row = {'quantization': method}
        
        # Average timing
        times = [r['timing']['mean_ms'] for r in results 
                if 'timing' in r and 'error' not in r['timing']]
        if times:
            row['avg_latency_ms'] = np.mean(times)
        
        # Average throughput
        tps = [r['tokens_per_sec'] for r in results if 'tokens_per_sec' in r]
        if tps:
            row['avg_tokens_per_sec'] = np.mean(tps)
        
        # Average quality
        f1s = [r['quality']['f1_mean'] for r in results if 'quality' in r]
        if f1s:
            row['avg_f1'] = np.mean(f1s)
        
        ems = [r['quality']['em_mean'] for r in results if 'quality' in r]
        if ems:
            row['avg_em'] = np.mean(ems)
        
        # Model size (should be same for both variants)
        sizes = [r['model_size']['size_gb'] for r in results if 'model_size' in r]
        if sizes:
            row['model_size_gb'] = sizes[0]
        
        # Average memory
        mems = [r['memory']['total_mb'] for r in results if 'memory' in r]
        if mems:
            row['avg_memory_mb'] = np.mean(mems)
        
        rows.append(row)
    
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values('quantization')
    
    return df

quant_comparison = aggregate_by_quantization()

In [85]:
print("\nQUANTIZATION METHOD COMPARISON")
print(quant_comparison.to_string(index=False))


QUANTIZATION METHOD COMPARISON
quantization  avg_latency_ms  avg_tokens_per_sec   avg_f1  avg_em  model_size_gb  avg_memory_mb
         awq     6663.905868            9.610561 0.109991   0.050       3.371388    4245.261230
        fp16     3780.082318           16.935270 0.103645   0.035      13.485551   16673.544922
        gptq     2436.287587           26.549877 0.050143   0.005       3.371388    4244.761230
         nf4     4850.323061           13.278879 0.092071   0.030       3.371388    4245.261230


Efficiency Rankings

In [86]:
def create_rankings() -> Dict[str, List[Tuple[str, float]]]:
    """Create rankings across different dimensions."""
    rankings = {}
    
    # Speed ranking
    speed_data = [(config, result['tokens_per_sec']) 
                  for config, result in analysis_results.items() 
                  if 'tokens_per_sec' in result]
    rankings['speed'] = sorted(speed_data, key=lambda x: x[1], reverse=True)
    
    # Memory efficiency ranking (lower is better)
    memory_data = [(config, result['memory']['total_mb']) 
                   for config, result in analysis_results.items() 
                   if 'memory' in result]
    rankings['memory'] = sorted(memory_data, key=lambda x: x[1])
    
    # Model size ranking (lower is better)
    size_data = [(config, result['model_size']['size_gb']) 
                 for config, result in analysis_results.items() 
                 if 'model_size' in result]
    rankings['size'] = sorted(size_data, key=lambda x: x[1])
    
    # Quality ranking
    quality_data = [(config, result['quality']['f1_mean']) 
                    for config, result in analysis_results.items() 
                    if 'quality' in result]
    rankings['quality'] = sorted(quality_data, key=lambda x: x[1], reverse=True)
    
    return rankings

rankings = create_rankings()

In [87]:
print("\nEFFICIENCY RANKINGS")

if 'speed' in rankings:
    print("\nBy Speed (tokens/sec, fastest first):")
    for i, (config, value) in enumerate(rankings['speed'], 1):
        print(f"  {i}. {config}: {value:.2f} tokens/sec")

if 'memory' in rankings:
    print("\nBy Memory Usage (MB, lowest first):")
    for i, (config, value) in enumerate(rankings['memory'], 1):
        print(f"  {i}. {config}: {value:.2f} MB")

if 'size' in rankings:
    print("\nBy Model Size (GB, smallest first):")
    for i, (config, value) in enumerate(rankings['size'], 1):
        print(f"  {i}. {config}: {value:.3f} GB")

if 'quality' in rankings:
    print("\nBy Quality (F1, highest first):")
    for i, (config, value) in enumerate(rankings['quality'], 1):
        print(f"  {i}. {config}: {value:.4f}")


EFFICIENCY RANKINGS

By Speed (tokens/sec, fastest first):
  1. gptq_instruct: 29.28 tokens/sec
  2. gptq_base: 23.82 tokens/sec
  3. fp16_instruct: 17.21 tokens/sec
  4. fp16_base: 16.66 tokens/sec
  5. nf4_instruct: 14.33 tokens/sec
  6. nf4_base: 12.22 tokens/sec
  7. awq_instruct: 9.86 tokens/sec
  8. awq_base: 9.36 tokens/sec

By Memory Usage (MB, lowest first):
  1. gptq_base: 4244.76 MB
  2. gptq_instruct: 4244.76 MB
  3. awq_base: 4245.26 MB
  4. awq_instruct: 4245.26 MB
  5. nf4_base: 4245.26 MB
  6. nf4_instruct: 4245.26 MB
  7. fp16_base: 16673.54 MB
  8. fp16_instruct: 16673.54 MB

By Model Size (GB, smallest first):
  1. awq_base: 3.371 GB
  2. awq_instruct: 3.371 GB
  3. nf4_base: 3.371 GB
  4. nf4_instruct: 3.371 GB
  5. gptq_base: 3.371 GB
  6. gptq_instruct: 3.371 GB
  7. fp16_base: 13.486 GB
  8. fp16_instruct: 13.486 GB

By Quality (F1, highest first):
  1. fp16_instruct: 0.1130
  2. awq_base: 0.1110
  3. awq_instruct: 0.1090
  4. nf4_instruct: 0.0972
  5. fp16_base

Compression and Speedup Analysis

In [88]:
def compute_relative_metrics() -> pd.DataFrame:
    """Compute metrics relative to FP16 baseline."""
    baseline_configs = ['fp16_base', 'fp16_instruct']
    
    # Find baseline metrics
    baseline_speed = []
    baseline_memory = []
    baseline_size = []
    
    for config in baseline_configs:
        if config in analysis_results:
            result = analysis_results[config]
            if 'tokens_per_sec' in result:
                baseline_speed.append(result['tokens_per_sec'])
            if 'memory' in result:
                baseline_memory.append(result['memory']['total_mb'])
            if 'model_size' in result:
                baseline_size.append(result['model_size']['size_gb'])
    
    avg_baseline_speed = np.mean(baseline_speed) if baseline_speed else None
    avg_baseline_memory = np.mean(baseline_memory) if baseline_memory else None
    avg_baseline_size = np.mean(baseline_size) if baseline_size else None
    
    rows = []
    for config_name, result in analysis_results.items():
        if config_name in baseline_configs:
            continue
        
        row = {
            'config': config_name,
            'quantization': result['quantization']
        }
        
        # Speedup
        if 'tokens_per_sec' in result and avg_baseline_speed:
            speedup = result['tokens_per_sec'] / avg_baseline_speed
            row['speedup'] = speedup
        
        # Memory reduction
        if 'memory' in result and avg_baseline_memory:
            memory_ratio = result['memory']['total_mb'] / avg_baseline_memory
            reduction = (1 - memory_ratio) * 100
            row['memory_reduction_pct'] = reduction
        
        # Compression ratio
        if 'model_size' in result and avg_baseline_size:
            compression = avg_baseline_size / result['model_size']['size_gb']
            row['compression_ratio'] = compression
        
        # Quality preservation
        if 'quality' in result:
            row['f1_score'] = result['quality']['f1_mean']
        
        rows.append(row)
    
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values('quantization')
    
    return df

relative_metrics = compute_relative_metrics()

In [89]:
print("\nRELATIVE TO FP16 BASELINE")
print(relative_metrics.to_string(index=False))


RELATIVE TO FP16 BASELINE
       config quantization  speedup  memory_reduction_pct  compression_ratio  f1_score
     awq_base          awq 0.552635             74.538940                4.0  0.110974
 awq_instruct          awq 0.582341             74.538940                4.0  0.109008
    gptq_base         gptq 1.406615             74.541939                4.0  0.049319
gptq_instruct         gptq 1.728839             74.541939                4.0  0.050967
     nf4_base          nf4 0.721777             74.538940                4.0  0.086981
 nf4_instruct          nf4 0.846416             74.538940                4.0  0.097161


Pareto Frontier Analysis

In [90]:
def compute_efficiency_scores() -> pd.DataFrame:
    """Compute composite efficiency scores."""
    rows = []
    
    for config_name, result in analysis_results.items():
        if 'fp16' in config_name:
            continue
        
        row = {'config': config_name}
        
        # Normalize metrics to [0, 1] where higher is better
        if 'tokens_per_sec' in result and 'quality' in result:
            speed = result['tokens_per_sec']
            quality = result['quality']['f1_mean']
            
            row['speed'] = speed
            row['quality'] = quality
            
            # Composite score (equal weighting)
            row['efficiency_score'] = (speed / 100) * quality
        
        if 'memory' in result and 'quality' in result:
            memory = result['memory']['total_mb']
            quality = result['quality']['f1_mean']
            
            # Memory efficiency (lower memory is better)
            memory_eff = 1 / (memory / 1000) if memory > 0 else 0
            row['memory_efficiency'] = memory_eff * quality
        
        rows.append(row)
    
    df = pd.DataFrame(rows)
    if not df.empty and 'efficiency_score' in df.columns:
        df = df.sort_values('efficiency_score', ascending=False)
    
    return df

efficiency_scores = compute_efficiency_scores()

In [91]:
print("\nEFFICIENCY SCORES")
print("Higher scores indicate better quality-efficiency trade-offs")
print(efficiency_scores.to_string(index=False))


EFFICIENCY SCORES
Higher scores indicate better quality-efficiency trade-offs
       config     speed  quality  efficiency_score  memory_efficiency
gptq_instruct 29.278356 0.050967          0.014922           0.012007
 nf4_instruct 14.334275 0.097161          0.013927           0.022887
    gptq_base 23.821398 0.049319          0.011748           0.011619
 awq_instruct  9.862105 0.109008          0.010750           0.025677
     nf4_base 12.223484 0.086981          0.010632           0.020489
     awq_base  9.359017 0.110974          0.010386           0.026141


Summary Statistics

In [92]:
def generate_summary() -> Dict:
    """Generate overall summary statistics."""
    summary = {}
    
    # Count configurations
    summary['total_configs'] = len(analysis_results)
    summary['quantization_methods'] = len(set(r['quantization'] for r in analysis_results.values()))
    
    # Speed range
    speeds = [r['tokens_per_sec'] for r in analysis_results.values() if 'tokens_per_sec' in r]
    if speeds:
        summary['speed_range_tps'] = {
            'min': float(np.min(speeds)),
            'max': float(np.max(speeds)),
            'range': float(np.max(speeds) - np.min(speeds))
        }
    
    # Memory range
    memories = [r['memory']['total_mb'] for r in analysis_results.values() if 'memory' in r]
    if memories:
        summary['memory_range_mb'] = {
            'min': float(np.min(memories)),
            'max': float(np.max(memories)),
            'range': float(np.max(memories) - np.min(memories))
        }
    
    # Size range
    sizes = [r['model_size']['size_gb'] for r in analysis_results.values() if 'model_size' in r]
    if sizes:
        summary['size_range_gb'] = {
            'min': float(np.min(sizes)),
            'max': float(np.max(sizes)),
            'compression': float(np.max(sizes) / np.min(sizes))
        }
    
    # Quality range
    qualities = [r['quality']['f1_mean'] for r in analysis_results.values() if 'quality' in r]
    if qualities:
        summary['quality_range_f1'] = {
            'min': float(np.min(qualities)),
            'max': float(np.max(qualities)),
            'range': float(np.max(qualities) - np.min(qualities))
        }
    
    return summary

summary = generate_summary()

In [93]:
print("\nSUMMARY STATISTICS")
for key, value in summary.items():
    if isinstance(value, dict):
        print(f"\n{key}:")
        for subkey, subvalue in value.items():
            print(f"  {subkey}: {subvalue:.4f}")
    else:
        print(f"{key}: {value}")


SUMMARY STATISTICS
total_configs: 8
quantization_methods: 4

speed_range_tps:
  min: 9.3590
  max: 29.2784
  range: 19.9193

memory_range_mb:
  min: 4244.7612
  max: 16673.5449
  range: 12428.7837

size_range_gb:
  min: 3.3714
  max: 13.4856
  compression: 4.0000

quality_range_f1:
  min: 0.0493
  max: 0.1130
  range: 0.0637


Save Results

In [94]:
output_data = {
    'comparison_table': comparison_df.to_dict(orient='records'),
    'quantization_comparison': quant_comparison.to_dict(orient='records'),
    'rankings': {k: [(c, float(v)) for c, v in vs] for k, vs in rankings.items()},
    'relative_metrics': relative_metrics.to_dict(orient='records'),
    'efficiency_scores': efficiency_scores.to_dict(orient='records'),
    'summary': summary,
    'full_analysis': analysis_results
}

output_file = OUTPUT_DIR / 'efficiency_analysis_complete.json'
with open(output_file, 'w') as f:
    json.dump(output_data, f, indent=2)

logger.info(f"Saved complete analysis to: {output_file}")

2025-12-28 23:24:28,288 - INFO - Saved complete analysis to: /kaggle/working/efficiency_analysis/efficiency_analysis_complete.json


In [95]:
csv_file = OUTPUT_DIR / 'comparison_table.csv'
comparison_df.to_csv(csv_file, index=False)
logger.info(f"Saved comparison table to: {csv_file}")

2025-12-28 23:24:28,306 - INFO - Saved comparison table to: /kaggle/working/efficiency_analysis/comparison_table.csv


In [96]:
print("\nANALYSIS COMPLETE")
print(f"Results saved to: {OUTPUT_DIR}")
print(f"  - efficiency_analysis_complete.json")
print(f"  - comparison_table.csv")


ANALYSIS COMPLETE
Results saved to: /kaggle/working/efficiency_analysis
  - efficiency_analysis_complete.json
  - comparison_table.csv
